# 10 — Projeto final: sistema de biblioteca

Este projeto integra coleções, funções, validação, exceções, arquivos JSON e orientação a objetos.

**Requisitos do sistema:**

- cadastrar livros e usuários;
- emprestar e devolver livros;
- impedir operações inválidas;
- pesquisar o catálogo;
- salvar e restaurar o estado em JSON.


## 1. Exceções do domínio

Nomes específicos deixam claro por que uma operação falhou e permitem tratamentos diferentes na interface.


In [1]:
class BibliotecaError(Exception):
    '''Classe-base para erros esperados da biblioteca.'''


class ItemNaoEncontradoError(BibliotecaError):
    pass


class LivroIndisponivelError(BibliotecaError):
    pass


class LimiteEmprestimosError(BibliotecaError):
    pass


## 2. Entidades

`Livro` e `Usuario` são dataclasses. O `default_factory` cria uma lista diferente para cada usuário.


In [2]:
from dataclasses import asdict, dataclass, field

@dataclass
class Livro:
    isbn: str
    titulo: str
    autor: str
    disponivel: bool = True

    def emprestar(self) -> None:
        if not self.disponivel:
            raise LivroIndisponivelError(f"{self.titulo} não está disponível")
        self.disponivel = False

    def devolver(self) -> None:
        self.disponivel = True


@dataclass
class Usuario:
    identificador: int
    nome: str
    isbns_emprestados: list[str] = field(default_factory=list)

    def pode_emprestar(self, limite: int) -> bool:
        return len(self.isbns_emprestados) < limite


## 3. Serviço principal

`Biblioteca` coordena as entidades. Dicionários permitem localizar livros e usuários diretamente por seus identificadores.


In [3]:
class Biblioteca:
    def __init__(self, nome: str, limite_por_usuario: int = 3):
        self.nome = nome
        self.limite_por_usuario = limite_por_usuario
        self._livros: dict[str, Livro] = {}
        self._usuarios: dict[int, Usuario] = {}

    def cadastrar_livro(self, livro: Livro) -> None:
        if livro.isbn in self._livros:
            raise BibliotecaError(f"ISBN já cadastrado: {livro.isbn}")
        self._livros[livro.isbn] = livro

    def cadastrar_usuario(self, usuario: Usuario) -> None:
        if usuario.identificador in self._usuarios:
            raise BibliotecaError(
                f"Usuário já cadastrado: {usuario.identificador}"
            )
        self._usuarios[usuario.identificador] = usuario

    def obter_livro(self, isbn: str) -> Livro:
        try:
            return self._livros[isbn]
        except KeyError as erro:
            raise ItemNaoEncontradoError(f"Livro não encontrado: {isbn}") from erro

    def obter_usuario(self, identificador: int) -> Usuario:
        try:
            return self._usuarios[identificador]
        except KeyError as erro:
            raise ItemNaoEncontradoError(
                f"Usuário não encontrado: {identificador}"
            ) from erro

    def emprestar(self, isbn: str, usuario_id: int) -> None:
        livro = self.obter_livro(isbn)
        usuario = self.obter_usuario(usuario_id)
        if not usuario.pode_emprestar(self.limite_por_usuario):
            raise LimiteEmprestimosError(
                f"{usuario.nome} atingiu o limite de empréstimos"
            )
        livro.emprestar()
        usuario.isbns_emprestados.append(isbn)

    def devolver(self, isbn: str, usuario_id: int) -> None:
        livro = self.obter_livro(isbn)
        usuario = self.obter_usuario(usuario_id)
        if isbn not in usuario.isbns_emprestados:
            raise BibliotecaError("Este empréstimo não pertence ao usuário")
        usuario.isbns_emprestados.remove(isbn)
        livro.devolver()

    def pesquisar(self, termo: str) -> list[Livro]:
        termo = termo.casefold().strip()
        return [
            livro
            for livro in self._livros.values()
            if termo in livro.titulo.casefold() or termo in livro.autor.casefold()
        ]

    def livros_disponiveis(self) -> list[Livro]:
        return [livro for livro in self._livros.values() if livro.disponivel]

    def __len__(self) -> int:
        return len(self._livros)


## 4. Carga inicial e operações


In [4]:
biblioteca = Biblioteca("Biblioteca Python", limite_por_usuario=2)

for livro in [
    Livro("978-1", "Python Fluente", "Luciano Ramalho"),
    Livro("978-2", "Automatize Tarefas Maçantes", "Al Sweigart"),
    Livro("978-3", "Código Limpo", "Robert C. Martin"),
]:
    biblioteca.cadastrar_livro(livro)

biblioteca.cadastrar_usuario(Usuario(1, "Ana"))
biblioteca.cadastrar_usuario(Usuario(2, "Caio"))

biblioteca.emprestar("978-1", 1)

print("Livros cadastrados:", len(biblioteca))
print("Empréstimos de Ana:", biblioteca.obter_usuario(1).isbns_emprestados)
print("Disponíveis:", [livro.titulo for livro in biblioteca.livros_disponiveis()])


Livros cadastrados: 3
Empréstimos de Ana: ['978-1']
Disponíveis: ['Automatize Tarefas Maçantes', 'Código Limpo']


## 5. Tratando uma regra de negócio

A aplicação captura somente erros que sabe apresentar ao usuário. O estado continua válido depois da tentativa.


In [5]:
try:
    biblioteca.emprestar("978-1", 2)
except LivroIndisponivelError as erro:
    print("Empréstimo recusado:", erro)

biblioteca.devolver("978-1", 1)
print("Disponível após devolução:", biblioteca.obter_livro("978-1").disponivel)


Empréstimo recusado: Python Fluente não está disponível
Disponível após devolução: True


## 6. Pesquisa


In [6]:
encontrados = biblioteca.pesquisar("python")
for livro in encontrados:
    print(f"{livro.titulo} — {livro.autor}")


Python Fluente — Luciano Ramalho


## 7. Persistência em JSON

Os métodos abaixo convertem objetos em dados simples. Em projetos maiores, essa responsabilidade poderia ser movida para uma classe de repositório.


In [7]:
import json
from pathlib import Path

def salvar_biblioteca(biblioteca: Biblioteca, caminho: Path) -> None:
    dados = {
        "nome": biblioteca.nome,
        "limite_por_usuario": biblioteca.limite_por_usuario,
        "livros": [asdict(livro) for livro in biblioteca._livros.values()],
        "usuarios": [asdict(usuario) for usuario in biblioteca._usuarios.values()],
    }
    caminho.write_text(
        json.dumps(dados, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def carregar_biblioteca(caminho: Path) -> Biblioteca:
    dados = json.loads(caminho.read_text(encoding="utf-8"))
    biblioteca = Biblioteca(dados["nome"], dados["limite_por_usuario"])
    for dados_livro in dados["livros"]:
        biblioteca.cadastrar_livro(Livro(**dados_livro))
    for dados_usuario in dados["usuarios"]:
        biblioteca.cadastrar_usuario(Usuario(**dados_usuario))
    return biblioteca


In [8]:
from tempfile import TemporaryDirectory

with TemporaryDirectory() as pasta:
    arquivo = Path(pasta) / "biblioteca.json"
    salvar_biblioteca(biblioteca, arquivo)
    copia = carregar_biblioteca(arquivo)

    print(copia.nome)
    print([livro.titulo for livro in copia.livros_disponiveis()])
    print(copia.obter_usuario(1))


Biblioteca Python
['Python Fluente', 'Automatize Tarefas Maçantes', 'Código Limpo']
Usuario(identificador=1, nome='Ana', isbns_emprestados=[])


## 8. Testes com `assert`

`assert` é útil para exemplos rápidos. Em aplicações reais, organize testes automatizados com `unittest` ou `pytest`.


In [9]:
biblioteca_teste = Biblioteca("Teste")
biblioteca_teste.cadastrar_livro(Livro("T-1", "Livro Teste", "Autora"))
biblioteca_teste.cadastrar_usuario(Usuario(10, "Pessoa Teste"))

biblioteca_teste.emprestar("T-1", 10)
assert not biblioteca_teste.obter_livro("T-1").disponivel
assert biblioteca_teste.obter_usuario(10).isbns_emprestados == ["T-1"]

biblioteca_teste.devolver("T-1", 10)
assert biblioteca_teste.obter_livro("T-1").disponivel
assert biblioteca_teste.obter_usuario(10).isbns_emprestados == []

print("Todos os testes passaram!")


Todos os testes passaram!


## Desafios para continuar

1. Registre a data prevista para devolução.
2. Crie uma fila de reservas para livros indisponíveis.
3. Separe as classes em módulos `.py`.
4. Substitua o JSON por SQLite.
5. Crie testes com `pytest`.

## Conclusão

Você percorreu os fundamentos da linguagem e aplicou POO em um domínio completo. A próxima etapa natural é transformar este notebook em um pacote Python com módulos, testes e persistência em banco de dados.
